# Notebook 03 — Feature Engineering

Build hand-crafted features from the preprocessed splits — no automated vectorizers (`TfidfVectorizer`, `CountVectorizer`). Every feature is constructed manually from `filtered_tokens`, per the assignment constraint documented in the repo README.

## 3.0 Load processed splits

Load `train.pkl`, `valid.pkl`, `test.pkl`, and `label_encoder.pkl` from `data/processed/` rather than re-running Notebook 02. The split, cleaning, and label encoding are already done and frozen — reloading them here keeps this notebook focused purely on feature engineering, and guarantees every downstream experiment works from the exact same train/val/test rows.

In [26]:
import pickle
import pandas as pd

In [ ]:
with open('../data/processed/train.pkl', 'rb') as f:
    train_df = pickle.load(f)

with open('../data/processed/valid.pkl', 'rb') as f:
    valid_df = pickle.load(f)

with open('../data/processed/test.pkl', 'rb') as f:
    test_df = pickle.load(f)

with open('../data/processed/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

**Result:** Loaded 4,147 train / 889 validation / 889 test rows, plus `label_encoder.pkl` (`{0: 'anger', 1: 'fear', 2: 'joy'}`). Row counts match Notebook 02's 70/15/15 split.

## 3.1 Build vocabulary from train tokens only

The vocabulary is built exclusively from `filtered_tokens` in the **train** split — never validation or test.

This matters because of leakage: if a word's presence in the validation or test set influences which words make it into the vocabulary, information about those sets has leaked into the feature-building process before a single model has been trained. That inflates validation/test metrics in a way that won't hold up on genuinely unseen data — the entire point of holding out those sets is to simulate data the pipeline has never seen.

Vocabulary size is capped by **minimum document frequency** — a word must appear in at least `MIN_DOC_FREQ` distinct sentences to be kept — rather than a fixed top-N. A frequency-based cutoff is directly justified by the measured word distribution below; a round top-N number would be arbitrary.

In [16]:
doc_freq = train_df['filtered_tokens'].apply(set).explode().value_counts()

> `.apply(set)` turns each sentence's token list into a set before exploding — `["ugh", "ugh", "ugh", "i", "hate", "this"]` becomes `{"ugh", "i", "hate", "this"}`, collapsing repeats within a sentence down to one. `.value_counts()` on the exploded result then counts how many *sentences* each word appears in, not how many total times — document frequency, not raw occurrence count.

In [17]:
print(f"Total vocabulary size (uncapped): {len(doc_freq)}")
print(f"Words appearing in exactly 1 doc: {(doc_freq == 1).sum()}")
print(f"Words appearing in exactly 2 docs: {(doc_freq == 2).sum()}")

Total vocabulary size (uncapped): 5936
Words appearing in exactly 1 doc: 3317
Words appearing in exactly 2 docs: 803


In [20]:
singleton_docs = (doc_freq == 1).sum()
print(f"Words appearing in exactly 1 sentence: {singleton_docs}")
print(f"Proportion of unique vocab that's singleton: {singleton_docs / len(doc_freq):.2%}")

Words appearing in exactly 1 sentence: 3317
Proportion of unique vocab that's singleton: 55.88%


In [21]:
singleton_words = set(doc_freq[doc_freq == 1].index)
rows_containing_only_singletons = train_df['filtered_tokens'].apply(
    lambda tokens: all(t in singleton_words for t in tokens) if len(tokens) > 0 else False
)
print(f"Sentences that would become fully empty if singletons dropped: {rows_containing_only_singletons.sum()}")

Sentences that would become fully empty if singletons dropped: 0


In [19]:
MIN_DOC_FREQ = 2
vocab = doc_freq[doc_freq >= MIN_DOC_FREQ].index.tolist()
print(f"Vocabulary size after min-doc-freq cutoff (>= {MIN_DOC_FREQ}): {len(vocab)}")

Vocabulary size after min-doc-freq cutoff (>= 2): 2619


**Result:** Document frequency (number of distinct sentences a word appears in) was used rather than raw occurrence count — a word repeated multiple times within a single sentence counts once, so rare words can't slip past the cutoff simply by being repeated in one place.

Singleton words (appearing in exactly one training sentence) make up 55.88% of the unique vocabulary (3,317 of 5,936), but cutting them at `MIN_DOC_FREQ = 2` leaves every sentence with at least one remaining word — zero training sentences become fully empty. Final vocabulary size after cutoff: **2,619** words, a 56% reduction in feature dimensionality for effectively no loss of usable training examples.

**Limitation:** because the vocabulary is built from train only, any word that appears in validation or test but never in train gets no column at all — it isn't "0" for some feature, it's simply absent from the check entirely. Out-of-vocabulary words are invisible to this feature set, not penalized by it.

## 3.2 Binary (presence/absence) word features

For each sentence, build one feature per vocabulary word: 1 if the word appears anywhere in the sentence's `filtered_tokens`, 0 otherwise. This captures whether a word shows up at all, independent of how many times — useful for short comments where a word rarely repeats, so presence alone may carry most of the signal.

In [24]:
binary_col = train_df['filtered_tokens'].apply(lambda tokens: [word in tokens for word in vocab])
binary_col.head(1)

4169    [True, False, False, False, False, False, Fals...
Name: filtered_tokens, dtype: object

In [29]:
binary_matrix = pd.DataFrame(binary_col.tolist(), columns=vocab, index=train_df.index)

In [33]:
binary_matrix.shape[0], binary_matrix.shape[1]

(4147, 2619)

In [ ]:
valid_binary_col = valid_df['filtered_tokens'].apply(lambda tokens: [word in tokens for word in vocab])
valid_matrix = pd.DataFrame(valid_binary_col.tolist(), columns=vocab, index=valid_df.index)

test_binary_col = test_df['filtered_tokens'].apply(lambda tokens: [word in tokens for word in vocab])
test_matrix = pd.DataFrame(test_binary_col.tolist(), columns=vocab, index=test_df.index)

**Result:** Binary feature matrix built for the train split — **4,147 rows × 2,619 columns**, one column per vocabulary word from 3.1, values `True`/`False` for presence in that sentence's `filtered_tokens`. Shape matches expectations (train row count × 3.1's cut-down vocabulary size).

## 3.3 Frequency (count) word features

Same vocabulary, but each feature now counts how many times the word appears in the sentence rather than just whether it appears. This preserves intensity information that binary features discard — e.g. a comment repeating "angry" three times may carry a stronger signal than one mentioning it once, even though both would look identical under the binary scheme.

**Result:**

## 3.4 Bigram features

Build word-pair (bigram) features from `filtered_tokens`, using the same binary/frequency logic as the unigram features above.

Bigrams matter specifically because of the negation-preservation decision made in Notebook 02, Step 2.6: negation words (`not`, `no`, `nor`, `never`, ...) were deliberately kept in the stopword removal step because they carry direct emotion signal. But a unigram bag-of-words treats `not` as just another isolated token — it loses *what* is being negated. Bigrams like `("not", "thankful")` capture the adjacency between the negation and the word it negates, which is exactly the signal Step 2.6 argued was worth preserving in the first place. Without bigrams, that earlier decision only partially pays off.

**Result:**

## 3.5 Sentence length feature

A single feature: the token count of each sentence's `filtered_tokens`.

Tested empirically across classes: mean length is 9.42 (anger), 9.33 (fear), and 9.75 (joy) tokens. The gap between classes is small — this is a weak signal, not a strong discriminator. It's included anyway because the cost is negligible: one extra scalar column per row, computed directly from data already in memory, with no risk of leakage and no meaningful impact on dimensionality.

**Result:**

## 3.6 Features tested and rejected

Not every feature idea tested during exploration made it into the final feature set. Documenting the rejected ones with evidence is as important as documenting the ones kept — it shows the decision was evidence-based, not arbitrary.

In [ ]:
train_df['lexical_diversity'] = train_df['filtered_tokens'].apply(
    lambda tokens: len(set(tokens)) / len(tokens) if len(tokens) > 0 else 0
)

diversity_by_emotion = train_df.groupby('Emotion')['lexical_diversity'].agg(['mean', 'median'])
diversity_by_emotion

### 3.6a Lexical diversity (unique tokens / total tokens)

**Hypothesis:** repetitive language might correlate with a specific emotion — e.g. if one class tends to repeat words more (or less) than another, the ratio of unique tokens to total tokens per sentence could carry discriminative signal.

**Result:** tested via `groupby('Emotion')` on the lexical diversity ratio. Mean is ~0.97 across all three classes (anger 0.973, fear 0.977, joy 0.973), and the median is exactly 1.0 for all three.

**Conclusion:** filtered sentences are short enough (~9 tokens average, per 3.5) that almost no repetition occurs regardless of class — there's no discriminative signal to extract. Rejected.

### 3.6b POS-tag ratio (e.g. adjective density per sentence)

**Hypothesis:** adjective-heavy sentences might correlate with a specific emotion — e.g. one class might lean more descriptive/adjective-laden than another.

**Reasoning for not fully testing this:** sentence length is already near-constant across classes (means 9.42 anger / 9.33 fear / 9.75 joy, tested in 3.5), and POS tag counts are bounded by sentence length — a sentence can't have more adjectives than it has tokens. Since length itself barely varies by class, a POS-ratio feature derived from it is unlikely to diverge meaningfully either.

This is a reasoned prediction, not an empirically confirmed one. Unlike 3.6a, this has not been tested against the data directly — it's flagged here as an item that could be tested later if time permits, rather than presented as an equivalent rejection.

In [ ]:
def adjective_ratio(pos_tags):
    if not pos_tags:
        return 0
    n_adj = sum(1 for _, tag in pos_tags if tag.startswith('JJ'))
    return n_adj / len(pos_tags)

train_df['adjective_ratio'] = train_df['POS_Tags'].apply(adjective_ratio)

adj_ratio_by_emotion = train_df.groupby('Emotion')['adjective_ratio'].agg(['mean', 'median'])
adj_ratio_by_emotion

**Synthesis:** length, diversity, and (reasoned) POS ratio all point in the same direction — structural/stylistic signal is weak in this dataset. This is likely because preprocessing already stripped punctuation, casing, and most function words, leaving content words to carry nearly all the discriminative signal. This is why the feature set leans on binary/frequency/bigram word features rather than structural ones.

## 3.7 Additional feature (TBD)

Open slot for one more hand-built feature — not yet decided. Candidates to consider: punctuation/capitalisation signals lost in Notebook 02's cleaning (e.g. exclamation count from the original `Comment`), POS-tag ratios (e.g. adjective density) from Notebook 02's `POS_Tags` column, or a lexicon-based score. Document whichever is chosen and why, same as every other feature above.

**Result:**

## 3.8 Assemble and save final feature matrices

Combine the hand-built feature columns (unigram binary/frequency, bigram binary/frequency, sentence length, and the Step 3.7 feature) into one final feature matrix per split, aligned with each split's `Label` column. Save the train/valid/test feature matrices to `data/processed/` so Notebook 04 (model training) can load them directly without recomputing features from scratch.

**Result:**